# Complexhibit — Exportación a RDF (OntoExhibit + CIDOC-CRM)

Convierte el gold standard anotado en GATE (100 catálogos) en un grafo de conocimiento en Turtle, según la ontología OntoExhibit.

OntoExhibit cubre exposiciones, actores, roles, instituciones, lugares, fechas y obras. Lo que no modela (técnica, material, dimensiones, tipo de obra, nº de inventario) se expresa con **CIDOC-CRM** (P32, P45, P43/E54, P2/E55, P1/E42). No necesita GPU.

| Manual de etiquetado | OntoExhibit / CIDOC-CRM |
|---|---|
| exhibition | `ox:Exhibition` · `ox:hasTitle` · `ox:hasTimeSpan` · `ox:hasVenue` · `ox:takesPlaceAt` · `ox:displays` |
| person + role | `ox:E21_Person` · `ox:hasRole` → `ox:Creator`, `ox:Lender`, `ox:Curator`, `ox:Editor`, `ox:Designer`, `ox:Member`… |
| institution + type/role | `ox:Institution` / `ox:Gallery` / `ox:Company` · `ox:hasOrganizer`, `ox:hasLender`, `ox:hasVenue` (host), `ox:Commissioner` (client) |
| placeName · address | `ox:Territorial_Entity` · `ox:Physical_Site` + `ox:address` |
| date (when, from/to) | `ox:Exact_Date` · `ox:Time_Range` + `ox:hasStartingDate` / `ox:hasEndingDate` |
| catalogueItem / work | `ox:Artwork` · `ox:hasCreation` → `ox:E65_Creation` (`ox:hasCreator`, `ox:hasTimeSpan`) |
| work type | `crm:P2_has_type` → `crm:E55_Type` |
| technique | `crm:P32_used_general_technique` → `crm:E55_Type` |
| material | `crm:P45_consists_of` → `crm:E57_Material` |
| dimensions (height, width, depth, weight, diameter) | `crm:P43_has_dimension` → `crm:E54_Dimension` (`P2` tipo, `P90` valor, `P91` unidad) |
| idno | `crm:P1_is_identified_by` → `crm:E42_Identifier` |
| catálogo | `ox:Catalog` · `crm:P67_refers_to` (cada entidad mencionada) |

Supuestos documentados: las entidades que aparecen fuera de un bloque se vinculan a la exposición principal del catálogo; si un catálogo es monográfico (un creador concentra ≥ 50 % de las menciones), ese creador se asigna a las obras cuya ficha no indica autor.

**Entrada:** un fichero ZIP con los ficheros XML anotados en GATE (carpeta `XML_GATE`). No necesita GPU.

**Salida:** `rdf_resultados.zip`, con el grafo completo (`gold_standard_ontoexhibit.ttl`), un fichero por catálogo y `resumen_rdf.csv` con las cifras principales.

In [ ]:
from google.colab import files
import os, glob, json, zipfile, shutil
print('Entrada: fichero ZIP con los XML de GATE (carpeta XML_GATE)')
subidos = files.upload()
shutil.rmtree('/content/xml', ignore_errors=True); os.makedirs('/content/xml')
for nombre in subidos:
    if nombre.lower().endswith('.zip'):
        zipfile.ZipFile(nombre).extractall('/content/xml')
    elif nombre.lower().endswith('.xml'):
        os.replace(nombre, '/content/xml/' + nombre)
for raiz, _, fs in os.walk('/content/xml'):
    for f in fs:
        if f.startswith('._'): os.remove(os.path.join(raiz, f))
shutil.rmtree('/content/xml/__MACOSX', ignore_errors=True)
CARPETA_XML = '/content/xml'
CARPETA_RDF = '/content/rdf'
shutil.rmtree(CARPETA_RDF, ignore_errors=True); os.makedirs(CARPETA_RDF)
print(len(glob.glob(CARPETA_XML + '/**/*.xml', recursive=True)), 'ficheros .xml')

In [ ]:
!pip -q install rdflib

In [ ]:
%%writefile complexhibit_ie.py
"""Complexhibit: extracción de información de catálogos en tres niveles.

Nivel 1  bloques:   catalogueItem (ITEM) y exhibition (EXHIB)        -> modelo BIO
Nivel 2  entidades: persName, title, institution, placeName, date...  -> modelo BIO
Nivel 3  detalle:   dimensiones y fechas                              -> reglas
                    roles y tipos (person.role, institution.type...)  -> clasificador

Este módulo contiene todo lo que no depende del modelo: lectura de GATE,
conversión a BIO, reglas, ejemplos para el clasificador de atributos,
decodificación de predicciones y montaje de la salida anidada.
"""
import json
import re
import xml.etree.ElementTree as ET
from fractions import Fraction
from pathlib import Path

# ----------------------------------------------------------------------------
# Esquema (según el Manual de etiquetado v1)
# ----------------------------------------------------------------------------
# El orden de cada diccionario fija la prioridad si dos anotaciones se solapan.
NIVEL2 = {
    "persName": "PER",
    "title": "TITLE",
    "institution": "ORG",
    "venue": "ORG",
    "placeName": "LOC",
    "date": "DATE",
    "technique": "TECH",
    "material": "TECH",
    "dimensions": "DIM",
    "address": "ADDR",
    "idno": "IDNO",
}
NIVEL1 = {
    "catalogueItem": "ITEM",
    "exhibition": "EXHIB",
}
# Atributos que aprende el clasificador del nivel 3: (tipo GATE, rasgo)
ATRIBUTOS = [
    ("person", "role"),
    ("institution", "type"),
    ("institution", "role"),
    ("work", "type"),
    ("technique", "type"),
]

# Reemplazo global erróneo detectado en el corpus: al renombrar valores de
# atributos (city -> poblation, material -> technique) se modificó también
# el texto. Se corrige solo el texto de cada palabra; las posiciones no cambian.
TOKEN_FIXES = [
    (re.compile(r"poblation"), "city"),
    (re.compile(r"techniqueity"), "materiality"),
    (re.compile(r"techniquely"), "materially"),
    (re.compile(r"detechniqueiz"), "dematerializ"),
]

TOKEN_RE = re.compile(r"\w+(?:[-'’]\w+)*|[^\w\s]", re.UNICODE)


# ----------------------------------------------------------------------------
# Lectura de textos etiquetados con GATE
# ----------------------------------------------------------------------------
def load_gate(path):
    """Devuelve (texto, anotaciones) con anotaciones = [(tipo, ini, fin, rasgos)]."""
    root = ET.parse(path).getroot()
    tw = root.find("TextWithNodes")
    text = tw.text or ""
    offsets = {}
    for node in tw:
        offsets[int(node.get("id"))] = len(text)
        text += node.tail or ""
    anns = []
    for aset in root.findall("AnnotationSet"):
        for a in aset.findall("Annotation"):
            s = offsets.get(int(a.get("StartNode")), int(a.get("StartNode")))
            e = offsets.get(int(a.get("EndNode")), int(a.get("EndNode")))
            feats = {f.find("Name").text: (f.find("Value").text or "") for f in a.findall("Feature")}
            if e > s:
                anns.append((a.get("Type"), s, e, feats))
    return text, anns


def fix_token(tok):
    for pat, rep in TOKEN_FIXES:
        tok = pat.sub(rep, tok)
    return tok


def tokenize(text):
    return [(m.group(), m.start(), m.end()) for m in TOKEN_RE.finditer(text)]


# ----------------------------------------------------------------------------
# Conversión a BIO
# ----------------------------------------------------------------------------
def resolve_overlaps(spans, prioridad):
    spans = sorted(spans, key=lambda x: (prioridad.index(x[2]), -(x[1] - x[0])))
    kept = []
    for s, e, t in spans:
        if all(e <= ks or s >= ke for ks, ke, _ in kept):
            kept.append((s, e, t))
    return sorted(kept)


def to_bio(tokens, spans, mapa):
    tags = ["O"] * len(tokens)
    for s, e, t in spans:
        inside = [i for i, (_, ts, te) in enumerate(tokens) if ts < e and te > s]
        for k, i in enumerate(inside):
            tags[i] = ("B-" if k == 0 else "I-") + mapa[t]
    return tags


def segment_bounds(tokens, text, tags=None, max_tokens=200):
    """Índices [ini, fin) de fragmentos: se corta en líneas en blanco o al llegar
    a max_tokens, sin partir nunca una entidad."""
    bounds, start = [], 0
    for i, (_, s, e) in enumerate(tokens):
        nxt = tokens[i + 1] if i + 1 < len(tokens) else None
        if nxt is None:
            bounds.append((start, i + 1))
            break
        inside_next = tags is not None and tags[i + 1].startswith("I-")
        gap = text[e:nxt[1]]
        if not inside_next and ("\n\n" in gap or i + 1 - start >= max_tokens):
            bounds.append((start, i + 1))
            start = i + 1
    return bounds


def documento_bio(path, mapa, max_tokens=200):
    text, anns = load_gate(path)
    spans = [(s, e, t) for t, s, e, _ in anns if t in mapa]
    tokens = tokenize(text)
    tags = to_bio(tokens, resolve_overlaps(spans, list(mapa)), mapa)
    filas = []
    for k, (a, b) in enumerate(segment_bounds(tokens, text, tags, max_tokens)):
        filas.append({
            "doc": Path(path).stem, "seg": k,
            "tokens": [fix_token(t[0]) for t in tokens[a:b]],
            "tags": tags[a:b],
        })
    return filas


def construir_bio(carpeta, mapa, salida, max_tokens=200):
    n, stats = 0, {}
    with open(salida, "w", encoding="utf-8") as fh:
        for p in sorted(Path(carpeta).rglob("*.xml")):
            try:
                filas = documento_bio(p, mapa, max_tokens)
            except ET.ParseError as err:
                print(f"AVISO: no se pudo leer {p.name}: {err}")
                continue
            for f in filas:
                fh.write(json.dumps(f, ensure_ascii=False) + "\n")
                for t in f["tags"]:
                    if t.startswith("B-"):
                        stats[t[2:]] = stats.get(t[2:], 0) + 1
            n += 1
    print(f"{n} documentos -> {salida}")
    print("Entidades por etiqueta:", dict(sorted(stats.items(), key=lambda x: -x[1])))
    return stats


# ----------------------------------------------------------------------------
# Nivel 3a: reglas de dimensiones
# ----------------------------------------------------------------------------
_NUM = r"\d+(?:[.,]\d+)?(?:\s+\d+/\d+)?|\d+/\d+"


def _valor(s):
    s = s.strip().replace(",", ".")
    try:
        return float(sum(Fraction(p) for p in s.split()))
    except (ValueError, ZeroDivisionError):
        return None


def parse_dimensions(texto, unidad_defecto=None):
    """'93 x 19 x 16' -> {'height': 93.0, 'width': 19.0, 'depth': 16.0, 'unit': ...}
    Convención de los catálogos: alto x ancho x fondo."""
    t = texto.lower()
    out = {}
    if re.search(r"\b(lbs?|pounds?)\b", t):
        out["unit"] = "libra"
    elif re.search(r"(?<![a-záéíóú])kg\b|kilo", t):
        out["unit"] = "kg"
    elif re.search(r"\bfeet\b|(?<![a-záéíóú])ft\b|\bpies\b", t):
        out["unit"] = "feet"
    elif re.search(r"(?<![a-záéíóú])cm\b", t):
        out["unit"] = "cm"
    elif re.search(r"(?<![a-záéíóú])mm\b", t):
        out["unit"] = "mm"
    elif re.search(r"(?<![a-záéíóú])m\b|metros", t):
        out["unit"] = "m"
    elif '"' in t or "inch" in t or "pulgada" in t or "in." in t:
        out["unit"] = "inches"
    if "unit" not in out and unidad_defecto:
        out["unit"] = unidad_defecto
    nums = re.findall(_NUM, t)
    if out.get("unit") in ("libra", "kg"):
        if nums:
            out["weight"] = _valor(nums[0])
        return out
    if re.search(r"\bdi[aá]m|ø|⌀", t) and nums:
        out["diameter"] = _valor(nums[0])
        return out
    rango = re.search(rf"({_NUM})\s+(?:to|a|-)\s+({_NUM})\s+(?:feet|ft|pies|cm|m)?\s*(?:in height|de alto|alto)", t)
    if rango:
        out["height"] = (_valor(rango.group(1)), _valor(rango.group(2)))
        return out
    if re.search(r"in height|de alto|\balto\b", t) and nums:
        out["height"] = _valor(nums[0])
        return out
    partes = re.split(r"\s*[x×]\s*", t)
    vals = [_valor(m.group()) for p in partes for m in [re.search(_NUM, p)] if m]
    for clave, v in zip(("height", "width", "depth"), vals):
        out[clave] = v
    return out


def unidad_documento(text):
    """Unidad declarada para todo el catálogo, p. ej. 'All dimensions are given
    in inches' o 'Medidas en centímetros'."""
    t = text.lower()
    m = re.search(r"\b(dimensions?|measurements?|medidas|dimensiones)\b[^.\n;]{0,30}?\b(?:in|en)\s+(inches|inch|cent[ií]met\w*|cm|metros|feet)\b", t)
    if not m:
        return None
    u = m.group(2)
    return "inches" if u.startswith("inch") else "feet" if u == "feet" else "m" if u == "metros" else "cm"


def unidad_mayoritaria(frases):
    """Unidad explícita más frecuente entre las dimensiones de un catálogo: se usa
    para las medidas que no indican unidad (p. ej. '19 5/8 x 29 13/16')."""
    cuenta = {}
    for f in frases:
        u = parse_dimensions(f).get("unit")
        if u and u not in ("libra", "kg"):   # solo unidades de longitud
            cuenta[u] = cuenta.get(u, 0) + 1
    return max(cuenta, key=cuenta.get) if cuenta else None


def unidad_por_defecto(text, frases):
    """Primero la unidad declarada en el catálogo ('dimensions are given in inches');
    si no la hay, la más frecuente entre sus medidas."""
    return unidad_documento(text) or unidad_mayoritaria(frases)


def dimensiones_oro(text, anns):
    """Pares (texto de la dimensión, valores anotados) para evaluar las reglas."""
    res = []
    for t, s, e, f in anns:
        if t != "dimensions":
            continue
        oro = {"unit": f.get("unit")} if f.get("unit") else {}
        for t2, s2, e2, _ in anns:
            if t2 in ("height", "width", "depth", "weight", "diameter") and s <= s2 and e2 <= e:
                v = text[s2:e2]
                m = re.match(rf"\s*({_NUM})\s+(?:to|a|-)\s+({_NUM})\s*$", v)
                oro[t2] = (_valor(m.group(1)), _valor(m.group(2))) if m else _valor(v)
        res.append((text[s:e], oro))
    return res


# ----------------------------------------------------------------------------
# Nivel 3b: reglas de fechas (formato ISO del manual: when / from-to)
# ----------------------------------------------------------------------------
MESES = {
    "jan": 1, "january": 1, "enero": 1, "ene": 1,
    "feb": 2, "february": 2, "febrero": 2,
    "mar": 3, "march": 3, "marzo": 3,
    "apr": 4, "april": 4, "abril": 4, "abr": 4,
    "may": 5, "mayo": 5,
    "jun": 6, "june": 6, "junio": 6,
    "jul": 7, "july": 7, "julio": 7,
    "aug": 8, "august": 8, "agosto": 8, "ago": 8,
    "sep": 9, "sept": 9, "september": 9, "septiembre": 9, "setiembre": 9,
    "oct": 10, "october": 10, "octubre": 10,
    "nov": 11, "november": 11, "noviembre": 11,
    "dec": 12, "december": 12, "diciembre": 12, "dic": 12,
}
_DIAS_MES = [31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]


def _ultimo_dia(y, m):
    if m == 2:
        return 29 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 28
    return _DIAS_MES[m - 1]


_RANGO_INCIERTO = re.compile(r"^\W*(\d{4})\s*[-–—]\s*(\d{2}|\d{4})\s*/\s*(\d{4})\W*$")
_ANIO_ABREV = re.compile(r"\b(\d{2})(\d{2})\s*[-–—]\s*(\d{2})(?![\d/])")


def parse_date(texto):
    t = texto.lower().replace("’", "'")
    # '1920-21/1987': fecha de creación incierta (1920 o 1921) y fecha posterior (p. ej., estampación)
    m = _RANGO_INCIERTO.match(t)
    if m:
        a, b, c = m.groups()
        b = a[:2] + b if len(b) == 2 else b
        return {"notBefore": a, "notAfter": b, "to": c}
    # '1969-71' -> '1969 - 1971'
    t = _ANIO_ABREV.sub(lambda m: f"{m.group(1)}{m.group(2)} - {m.group(1)}{m.group(3)}", t)
    toks = re.findall(r"[a-záéíóúñ]+|\d+", t)
    fechas = []  # lista de [dia, mes, año] en orden de aparición
    cur = [None, None, None]
    for tok in toks:
        if tok.isdigit():
            n = int(tok)
            if len(tok) == 4 and 1000 <= n <= 2100:
                cur[2] = n
                fechas.append(cur)
                cur = [None, None, None]
            elif n <= 31:
                if cur[0] is not None:  # nuevo día sin año: cerrar el anterior
                    fechas.append(cur)
                    cur = [None, None, None]
                cur[0] = n
        elif tok in MESES:
            if cur[1] is not None:
                fechas.append(cur)
                cur = [None, None, None]
            cur[1] = MESES[tok]
    if cur != [None, None, None]:
        fechas.append(cur)
    # formatos numéricos dd/mm/aaaa
    num = [x for x in re.findall(r"(\d{1,2})[/.-](\d{1,2})[/.-](\d{4})", t) if 1 <= int(x[1]) <= 12]
    if num:
        fechas = [[int(d), int(m), int(y)] for d, m, y in num]
    fechas = [f for f in fechas if any(x is not None for x in f)]
    if not fechas:
        return {}
    # el año (y el mes) se propagan hacia atrás: "9 January-6 March, 1998"
    for i in range(len(fechas) - 2, -1, -1):
        if fechas[i][2] is None:
            fechas[i][2] = fechas[i + 1][2]
            if fechas[i][1] is None:
                fechas[i][1] = fechas[i + 1][1]
    if any(f[2] is None for f in fechas):
        return {}
    # el mes se propaga hacia delante en rangos de días: 'June 20-26, 1963'
    for i in range(1, len(fechas)):
        if fechas[i][1] is None and fechas[i][0] is not None and fechas[i - 1][1] is not None:
            fechas[i][1] = fechas[i - 1][1]

    def iso(f, fin=False):
        d, m, y = f
        if m is None:
            return f"{y:04d}" if not fin else f"{y:04d}"
        if d is None:
            d = _ultimo_dia(y, m) if fin else 1
        return f"{y:04d}-{m:02d}-{d:02d}"

    if len(fechas) == 1:
        f = fechas[0]
        if f[1] is None:
            return {"when": f"{f[2]:04d}"}
        if f[0] is None:  # mes y año: intervalo del mes
            return {"from": iso(f), "to": iso(f, fin=True)}
        return {"when": iso(f)}
    return {"from": iso(fechas[0]), "to": iso(fechas[-1], fin=True)}


def fechas_oro(text, anns):
    res = []
    for t, s, e, f in anns:
        if t == "date":
            oro = {k: v for k, v in f.items() if k in ("when", "from", "to", "notBefore", "notAfter")}
            if oro:
                res.append((text[s:e], oro))
    return res


def evaluar_reglas(carpeta, docs=None):
    """Exactitud de las reglas frente a los valores anotados a mano."""
    dim_ok = dim_n = fec_ok = fec_n = 0
    errores = {"dimensiones": [], "fechas": []}
    for p in sorted(Path(carpeta).rglob("*.xml")):
        if docs is not None and p.stem not in docs:
            continue
        try:
            text, anns = load_gate(p)
        except ET.ParseError:
            continue
        pares = dimensiones_oro(text, anns)
        udoc = unidad_por_defecto(text, [f for f, _ in pares])
        for frase, oro in pares:
            pred = parse_dimensions(frase, udoc)
            campos = [k for k in oro if k != "unit"]
            ok = all(pred.get(k) == oro[k] for k in campos) and (not oro.get("unit") or pred.get("unit") == oro["unit"])
            dim_n += 1
            dim_ok += ok
            if not ok and len(errores["dimensiones"]) < 30:
                errores["dimensiones"].append({"texto": frase, "oro": oro, "regla": pred, "doc": p.stem})
        for frase, oro in fechas_oro(text, anns):
            pred = parse_date(frase)
            ok = pred == oro
            fec_n += 1
            fec_ok += ok
            if not ok and len(errores["fechas"]) < 30:
                errores["fechas"].append({"texto": frase, "oro": oro, "regla": pred, "doc": p.stem})
    return {
        "dimensiones": {"n": dim_n, "exactitud": dim_ok / dim_n if dim_n else None},
        "fechas": {"n": fec_n, "exactitud": fec_ok / fec_n if fec_n else None},
        "errores": errores,
    }


# ----------------------------------------------------------------------------
# Nivel 3c: ejemplos para el clasificador de roles y tipos
# ----------------------------------------------------------------------------
VENTANA = 250  # caracteres de contexto a cada lado


def con_marcas(text, s, e, ventana=VENTANA):
    a, b = max(0, s - ventana), min(len(text), e + ventana)
    frag = text[a:s] + " [E] " + text[s:e] + " [/E] " + text[e:b]
    frag = " ".join(fix_token(w) for w in frag.split())
    return frag


def ejemplos_atributos(carpeta):
    filas = []
    for p in sorted(Path(carpeta).rglob("*.xml")):
        try:
            text, anns = load_gate(p)
        except ET.ParseError:
            continue
        for t, s, e, f in anns:
            for tipo, rasgo in ATRIBUTOS:
                if t == tipo and f.get(rasgo, "").strip():
                    valor = re.sub(r"\s*,\s*", ", ", f[rasgo].strip().lstrip("_"))
                    filas.append({
                        "doc": p.stem, "atributo": f"{tipo}.{rasgo}",
                        "texto": f"{tipo}.{rasgo}: " + con_marcas(text, s, e),
                        "valor": valor,
                    })
    return filas


# ----------------------------------------------------------------------------
# Inferencia de etiquetas por palabra a fragmentos y a una salida anidada
# ----------------------------------------------------------------------------
def bio_a_spans(tokens, tags):
    """tokens = [(palabra, ini, fin)], tags = ['B-PER', ...] -> [(ini, fin, 'PER')]"""
    spans, cur = [], None
    for (w, s, e), tag in zip(tokens, tags):
        if tag.startswith("B-") or (tag.startswith("I-") and (cur is None or cur[2] != tag[2:])):
            if cur:
                spans.append(tuple(cur))
            cur = [s, e, tag[2:]]
        elif tag.startswith("I-") and cur:
            cur[1] = e
        else:
            if cur:
                spans.append(tuple(cur))
            cur = None
    if cur:
        spans.append(tuple(cur))
    return spans


def montar_salida(text, spans_n1, spans_n2, atributos=None):
    """Combina niveles 1, 2 y 3 en una estructura anidada (al estilo del manual).
    atributos: dict {(ini, fin): {'person.role': 'creator', ...}}"""
    atributos = atributos or {}
    udoc = unidad_por_defecto(text, [text[s:e] for s, e, lab in spans_n2 if lab == "DIM"])

    def entidad(s, e, lab):
        d = {"tipo": lab, "texto": text[s:e].strip(), "ini": s, "fin": e}
        if lab == "DIM":
            d["valores"] = parse_dimensions(text[s:e], udoc)
        if lab == "DATE":
            d["valores"] = parse_date(text[s:e])
        if (s, e) in atributos:
            d["atributos"] = atributos[(s, e)]
        return d

    bloques = []
    for s, e, lab in spans_n1:
        hijos = [entidad(s2, e2, l2) for s2, e2, l2 in spans_n2 if s <= s2 and e2 <= e]
        b = {"tipo": {"ITEM": "catalogueItem", "EXHIB": "exhibition"}.get(lab, lab),
             "texto": text[s:e].strip()[:300], "ini": s, "fin": e, "contenido": hijos}
        if (s, e) in atributos:
            b["atributos"] = atributos[(s, e)]
        bloques.append(b)
    dentro = {(s2, e2) for s, e, _ in spans_n1 for s2, e2, _ in spans_n2 if s <= s2 and e2 <= e}
    sueltas = [entidad(s, e, l) for s, e, l in spans_n2 if (s, e) not in dentro]
    return {"bloques": bloques, "entidades_fuera_de_bloques": sueltas}


In [ ]:
%%writefile complexhibit_rdf.py
"""Exportación a RDF (Turtle) según OntoExhibit, completada con CIDOC-CRM.

Convierte en RDF:
  * la salida anidada de los tres niveles (JSON de complexhibit_ie.montar_salida);
  * el gold standard anotado en GATE .

Correspondencias principales
  exhibition            -> ox:Exhibition  (hasTitle, hasTimeSpan, hasVenue, takesPlaceAt,
                                           hasOrganizer, hasLender, hasCuratorInvolved,
                                           hasExhibitingActant, displays)
  catalogueItem / work  -> ox:Artwork     (hasTitle, hasCreation -> E65_Creation)
  persName + role       -> ox:E21_Person  + ox:hasRole -> rol (ox:Creator, ox:Lender...)
  institution + type    -> ox:Institution / ox:Gallery / ox:Company ... + rol
  placeName             -> ox:Territorial_Entity
  address               -> ox:Physical_Site con ox:address
  date                  -> ox:Exact_Date / ox:Time_Range (hasStartingDate, hasEndingDate)
  técnica               -> crm:P32_used_general_technique -> crm:E55_Type
  material              -> crm:P45_consists_of -> crm:E57_Material
  dimensiones           -> crm:P43_has_dimension -> crm:E54_Dimension (P2, P90, P91)
  tipo de obra          -> crm:P2_has_type -> crm:E55_Type
  nº de inventario      -> crm:P1_is_identified_by -> crm:E42_Identifier
  catálogo              -> ox:Catalog, que crm:P67_refers_to cada entidad mencionada
"""
import collections
import re
import unicodedata
from pathlib import Path

import complexhibit_ie as ie

OX = "https://w3id.org/OntoExhibit#"
CRM = "http://www.cidoc-crm.org/cidoc-crm/"
BASE_DATOS = "https://w3id.org/OntoExhibit/datos/"   # cambiar si se publica en otro espacio

PREFIJOS = f"""@prefix ox: <{OX}> .
@prefix crm: <{CRM}> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix d: <{BASE_DATOS}> .

"""

# rol de persona segun guia de etiquetado -> clase de rol de OntoExhibit
ROL_PERSONA = {
    "creator": "Creator", "lender": "Lender", "owner": "Owner", "funder": "Funder",
    "exhibitioncurator": "Curator", "museumcurator": "Curator", "curator": "Curator",
    "assistant": "Assistant", "editor": "Editor", "designer": "Designer",
    "photographer": "Creator", "trustee": "Member", "author": "Author",
}
# rol de institución segun guia de etiquetado -> clase de rol de OntoExhibit ('host' se trata como sede)
ROL_INSTITUCION = {
    "organiser": "Organizer", "organizer": "Organizer", "lender": "Lender", "owner": "Owner",
    "funder": "Funder", "client": "Commissioner", "depository": "Owner",
}
# tipo de institución segun guia de etiquetado -> clase de OntoExhibit
TIPO_INSTITUCION = {
    "gallery": "Gallery", "artgallery": "Gallery", "company": "Company",
}


# ----------------------------------------------------------------------------
def slug(texto, maximo=60):
    t = unicodedata.normalize("NFKD", texto.lower()).encode("ascii", "ignore").decode()
    t = re.sub(r"[^a-z0-9]+", "-", t).strip("-")
    return t[:maximo] or "x"


def lit(texto, tipo=None):
    t = str(texto).replace("\\", "\\\\").replace('"', '\\"').replace("\n", " ").replace("\r", " ")
    t = re.sub(r"\s+", " ", t).strip()
    return f'"{t}"^^xsd:{tipo}' if tipo else f'"{t}"'


def valores_lista(v):
    return [x.strip().lstrip("_").lower() for x in re.split(r"[,+]", v or "") if x.strip()]


class Grafo:
    def __init__(self):
        self.lineas = []
        self.n = 0
        self.doc = "doc"

    def t(self, s, p, o):
        self.lineas.append(f"{s} {p} {o} .")

    def nuevo(self, prefijo):
        self.n += 1
        return f"d:{prefijo}-{self.doc}-{self.n}"

    def turtle(self):
        return PREFIJOS + "\n".join(dict.fromkeys(self.lineas)) + "\n"


# ----------------------------------------------------------------------------
def _fecha(g, valores, texto):
    """Devuelve el nodo de tiempo (Exact_Date o Time_Range) o None."""
    if not valores:
        return None
    if "when" in valores:
        n = g.nuevo("fecha")
        tipo = "gYear" if len(valores["when"]) == 4 else "date"
        g.t(n, "a", "ox:Exact_Date")
        g.t(n, "rdfs:label", lit(valores["when"], tipo))
        g.t(n, "crm:P3_has_note", lit(texto))
        return n
    if "from" in valores or "to" in valores:
        r = g.nuevo("intervalo")
        g.t(r, "a", "ox:Time_Range")
        g.t(r, "crm:P3_has_note", lit(texto))
        for clave, prop in (("from", "ox:hasStartingDate"), ("to", "ox:hasEndingDate")):
            if clave in valores:
                n = g.nuevo("fecha")
                g.t(n, "a", "ox:Exact_Date")
                g.t(n, "rdfs:label", lit(valores[clave], "date" if len(valores[clave]) == 10 else "gYear"))
                g.t(r, prop, n)
        return r
    return None


def _persona(g, ent):
    p = f"d:persona-{slug(ent['texto'])}"
    g.t(p, "a", "ox:E21_Person")
    g.t(p, "ox:person_name", lit(ent["texto"]))
    g.t(p, "rdfs:label", lit(ent["texto"]))
    return p


def _institucion(g, ent):
    i = f"d:institucion-{slug(ent['texto'])}"
    tipos = valores_lista(ent.get("atributos", {}).get("institution.type"))
    clase = next((TIPO_INSTITUCION[t] for t in tipos if t in TIPO_INSTITUCION), "Institution")
    g.t(i, "a", f"ox:{clase}")
    g.t(i, "ox:name", lit(ent["texto"]))
    g.t(i, "rdfs:label", lit(ent["texto"]))
    for t in tipos:
        if t not in TIPO_INSTITUCION:
            g.t(i, "crm:P2_has_type", f"d:tipo-institucion-{slug(t)}")
            g.t(f"d:tipo-institucion-{slug(t)}", "a", "crm:E55_Type")
            g.t(f"d:tipo-institucion-{slug(t)}", "rdfs:label", lit(t))
    return i


def _rol(g, actante, clase):
    r = g.nuevo("rol")
    g.t(r, "a", f"ox:{clase}")
    g.t(actante, "ox:hasRole", r)
    g.t(r, "ox:isRoleOf", actante)
    return r


def _lugar(g, ent):
    l = f"d:lugar-{slug(ent['texto'])}"
    g.t(l, "a", "ox:Territorial_Entity")
    g.t(l, "rdfs:label", lit(ent["texto"]))
    return l


def _tipo(g, espacio, valor):
    u = f"d:{espacio}-{slug(valor)}"
    g.t(u, "a", "crm:E55_Type")
    g.t(u, "rdfs:label", lit(valor))
    return u


def _dimensiones(g, obra, valores, texto):
    unidad = valores.get("unit")
    for clave in ("height", "width", "depth", "weight", "diameter"):
        if clave not in valores or valores[clave] is None:
            continue
        dmn = g.nuevo("dimension")
        g.t(obra, "crm:P43_has_dimension", dmn)
        g.t(dmn, "a", "crm:E54_Dimension")
        g.t(dmn, "crm:P2_has_type", _tipo(g, "tipo-dimension", clave))
        v = valores[clave]
        if isinstance(v, (tuple, list)):
            g.t(dmn, "crm:P90a_has_lower_value_limit", lit(v[0], "decimal"))
            g.t(dmn, "crm:P90b_has_upper_value_limit", lit(v[1], "decimal"))
        else:
            g.t(dmn, "crm:P90_has_value", lit(v, "decimal"))
        if unidad:
            u = f"d:unidad-{slug(unidad)}"
            g.t(u, "a", "crm:E58_Measurement_Unit")
            g.t(u, "rdfs:label", lit(unidad))
            g.t(dmn, "crm:P91_has_unit", u)
        g.t(dmn, "crm:P3_has_note", lit(texto))


def _titulo(g, entidad_uri, texto):
    t = g.nuevo("titulo")
    g.t(t, "a", "ox:Title")
    g.t(t, "rdfs:label", lit(texto))
    g.t(entidad_uri, "ox:hasTitle", t)
    g.t(t, "ox:isTitleOf", entidad_uri)


# ----------------------------------------------------------------------------
def _exposicion(g, b, cat):
    e = g.nuevo("exposicion")
    g.t(e, "a", "ox:Exhibition")
    g.t(cat, "crm:P67_refers_to", e)
    titulo_hecho = fecha_hecha = False
    for ent in b["contenido"]:
        lab, at = ent["tipo"], ent.get("atributos", {})
        if lab == "TITLE" and not titulo_hecho:
            _titulo(g, e, ent["texto"]); g.t(e, "rdfs:label", lit(ent["texto"])); titulo_hecho = True
        elif lab == "DATE" and not fecha_hecha:
            n = _fecha(g, ent.get("valores"), ent["texto"])
            if n:
                g.t(e, "ox:hasTimeSpan", n); fecha_hecha = True
        elif lab == "LOC":
            g.t(e, "ox:takesPlaceAt", _lugar(g, ent))
        elif lab == "ADDR":
            s = g.nuevo("sitio"); g.t(s, "a", "ox:Physical_Site"); g.t(s, "ox:address", lit(ent["texto"]))
            g.t(e, "ox:hasVenue", s)
        elif lab == "ORG":
            _vincular_institucion(g, e, _institucion(g, ent), ent)
        elif lab == "PER":
            _vincular_persona(g, e, _persona(g, ent), ent)
    return e


def _vincular_institucion(g, expo, inst, ent):
    roles = valores_lista(ent.get("atributos", {}).get("institution.role"))
    if expo is None:
        return
    for r in roles:
        if r == "host":
            g.t(expo, "ox:hasVenue", inst)
        elif r in ROL_INSTITUCION:
            rol = _rol(g, inst, ROL_INSTITUCION[r])
            prop = {"Organizer": "ox:hasOrganizer", "Lender": "ox:hasLender"}.get(ROL_INSTITUCION[r])
            if prop:
                g.t(expo, prop, rol)


def _vincular_persona(g, expo, per, ent):
    rol_txt = ent.get("atributos", {}).get("person.role", "").strip().lower()
    clase = ROL_PERSONA.get(rol_txt.replace("_", ""))
    if not clase:
        return None
    rol = _rol(g, per, clase)
    if expo is not None:
        if clase == "Curator":
            g.t(expo, "ox:hasCuratorInvolved", rol)
        elif clase == "Lender":
            g.t(expo, "ox:hasLender", rol)
        elif clase == "Creator":
            g.t(expo, "ox:hasExhibitingActant", per)
    return rol


def _obra(g, b, cat, expo, creadores_doc):
    o = g.nuevo("obra")
    g.t(o, "a", "ox:Artwork")
    g.t(cat, "crm:P67_refers_to", o)
    if expo is not None:
        g.t(expo, "ox:displays", o)
        g.t(o, "ox:isDisplayedAt", expo)
    tipo_obra = b.get("atributos", {}).get("work.type")
    if tipo_obra:
        g.t(o, "crm:P2_has_type", _tipo(g, "tipo-obra", tipo_obra))
    creacion = None
    titulo_hecho = False
    hay_creador = False
    for ent in b["contenido"]:
        lab = ent["tipo"]
        if lab == "TITLE" and not titulo_hecho:
            _titulo(g, o, ent["texto"]); g.t(o, "rdfs:label", lit(ent["texto"])); titulo_hecho = True
        elif lab == "DATE":
            n = _fecha(g, ent.get("valores"), ent["texto"])
            if n:
                creacion = creacion or _creacion(g, o)
                g.t(creacion, "ox:hasTimeSpan", n)
        elif lab == "TECH":
            tt = ent.get("atributos", {}).get("technique.type")
            for v in (valores_lista(tt) if tt else [ent["texto"]]):
                g.t(o, "crm:P32_used_general_technique", _tipo(g, "tecnica", v))
            g.t(o, "crm:P3_has_note", lit("Técnica: " + ent["texto"]))
        elif lab == "MAT":
            m = f"d:material-{slug(ent['texto'])}"
            g.t(m, "a", "crm:E57_Material"); g.t(m, "rdfs:label", lit(ent["texto"]))
            g.t(o, "crm:P45_consists_of", m)
        elif lab == "DIM":
            _dimensiones(g, o, ent.get("valores") or {}, ent["texto"])
        elif lab == "IDNO":
            i = g.nuevo("identificador"); g.t(i, "a", "crm:E42_Identifier"); g.t(i, "rdfs:label", lit(ent["texto"]))
            g.t(o, "crm:P1_is_identified_by", i)
        elif lab == "PER":
            per = _persona(g, ent)
            rol = _vincular_persona(g, None, per, ent)
            if rol and ent.get("atributos", {}).get("person.role", "").lower() == "creator":
                creacion = creacion or _creacion(g, o)
                g.t(creacion, "ox:hasCreator", rol); hay_creador = True
            elif rol and ent.get("atributos", {}).get("person.role", "").lower() == "owner":
                g.t(o, "ox:hasOwner", rol)
        elif lab == "ORG":
            inst = _institucion(g, ent)
            if "owner" in valores_lista(ent.get("atributos", {}).get("institution.role")):
                g.t(o, "ox:hasOwner", _rol(g, inst, "Owner"))
        elif lab == "LOC":
            creacion = creacion or _creacion(g, o)
            g.t(creacion, "ox:hasPlaceOfCreation", _lugar(g, ent))
    # en catálogos monográficos el autor suele aparecer solo fuera de la ficha
    if not hay_creador and len(creadores_doc) == 1:
        creacion = creacion or _creacion(g, o)
        g.t(creacion, "ox:hasCreator", creadores_doc[0])
    return o


def _creacion(g, obra):
    c = g.nuevo("creacion")
    g.t(c, "a", "ox:E65_Creation")
    g.t(obra, "ox:hasCreation", c)
    g.t(c, "ox:isCreationOf", obra)
    return c


def salida_a_turtle(salida, doc_id, g=None):
    """salida = dict de complexhibit_ie.montar_salida. Devuelve el objeto Grafo."""
    g = g or Grafo()
    g.doc = slug(doc_id, 40)
    cat = f"d:catalogo-{slug(doc_id)}"
    g.t(cat, "a", "ox:Catalog")
    g.t(cat, "rdfs:label", lit(doc_id))
    expos = [b for b in salida["bloques"] if b["tipo"] == "exhibition"]
    obras = [b for b in salida["bloques"] if b["tipo"] == "catalogueItem"]
    expo_uris = [_exposicion(g, b, cat) for b in expos]
    principal = expo_uris[0] if expo_uris else None
    # entidades fuera de bloques: se vinculan a la exposición principal del catálogo
    for ent in salida["entidades_fuera_de_bloques"]:
        lab = ent["tipo"]
        if lab == "PER":
            per = _persona(g, ent); g.t(cat, "crm:P67_refers_to", per)
            _vincular_persona(g, principal, per, ent)
        elif lab == "ORG":
            inst = _institucion(g, ent); g.t(cat, "crm:P67_refers_to", inst)
            _vincular_institucion(g, principal, inst, ent)
        elif lab == "LOC":
            g.t(cat, "crm:P67_refers_to", _lugar(g, ent))
    # autor principal del catálogo: el creador que concentra la mayoría de las menciones
    # (fuera de las fichas); se asigna a las obras cuya ficha no indica autor
    creadores = collections.Counter(
        slug(ent["texto"])
        for ent in salida["entidades_fuera_de_bloques"] + [x for b in expos for x in b["contenido"]]
        if ent["tipo"] == "PER" and ent.get("atributos", {}).get("person.role", "").lower() == "creator")
    creadores_rol = []
    if creadores:
        nombre, n = creadores.most_common(1)[0]
        if n / sum(creadores.values()) >= 0.5:
            creadores_rol = [_rol(g, f"d:persona-{nombre}", "Creator")]
    for b in obras:
        _obra(g, b, cat, principal, creadores_rol)
    return g


# ----------------------------------------------------------------------------
# Gold standard -> misma estructura que la salida del sistema
# ----------------------------------------------------------------------------
NIVEL2_GOLD = dict(ie.NIVEL2)
NIVEL2_GOLD["material"] = "MAT"   # en el gold se distingue material de técnica


def gold_a_salida(path):
    text, anns = ie.load_gate(path)
    # en el gold se conservan todas las anotaciones, tambien las que se superponen (p. ej., título y material
    # sobre el mismo texto). Para entrenar, la conversión a BIO deja una sola etiqueta por palabra; aquí no es necesario.
    sp1 = sorted({(s, e, ie.NIVEL1[t]) for t, s, e, _ in anns if t in ie.NIVEL1})
    sp2 = sorted({(s, e, NIVEL2_GOLD[t]) for t, s, e, _ in anns if t in NIVEL2_GOLD})
    atributos = {}

    def buscar(tipo, s, e, rasgo, contenido=False):
        for t, a, b, f in anns:
            if t != tipo or not f.get(rasgo):
                continue
            if (a <= s and e <= b) if not contenido else (s <= a and b <= e):
                return f[rasgo].strip().lstrip("_")
        return None

    for s, e, lab in sp2:
        d = {}
        if lab == "PER":
            v = buscar("person", s, e, "role"); d.update({"person.role": v} if v else {})
        if lab == "ORG":
            for r in ("type", "role"):
                v = buscar("institution", s, e, r); d.update({f"institution.{r}": v} if v else {})
        if lab == "TECH":
            v = buscar("technique", s, e, "type"); d.update({"technique.type": v} if v else {})
        if d:
            atributos[(s, e)] = d
    for s, e, lab in sp1:
        if lab == "ITEM":
            v = buscar("work", s, e, "type", contenido=True)
            if v:
                atributos[(s, e)] = {"work.type": v}
    return text, ie.montar_salida(text, sp1, sp2, atributos)


def exportar_gold(carpeta, salida_ttl, por_documento=None):
    g = Grafo()
    n = 0
    for p in sorted(Path(carpeta).rglob("*.xml")):
        try:
            _, sal = gold_a_salida(p)
        except Exception as err:  # noqa: BLE001
            print(f"AVISO {p.name}: {err}")
            continue
        salida_a_turtle(sal, p.stem, g)
        if por_documento:
            Path(por_documento).mkdir(parents=True, exist_ok=True)
            Path(por_documento, p.stem + ".ttl").write_text(salida_a_turtle(sal, p.stem).turtle(), encoding="utf-8")
        n += 1
    Path(salida_ttl).write_text(g.turtle(), encoding="utf-8")
    print(f"{n} catálogos -> {salida_ttl} ({len(set(g.lineas))} tripletas)")
    return g


## 1. Gold standard → RDF

In [ ]:
import complexhibit_rdf as rdf
rdf.exportar_gold(CARPETA_XML, CARPETA_RDF + '/gold_standard_ontoexhibit.ttl', CARPETA_RDF + '/gold_por_catalogo')

## 2. Validación y resumen

In [ ]:
import rdflib
for f in sorted(glob.glob(CARPETA_RDF + '/*.ttl')):
    G = rdflib.Graph().parse(f, format='turtle')
    print(os.path.basename(f), '→', len(G), 'tripletas')

G = rdflib.Graph().parse(CARPETA_RDF + '/gold_standard_ontoexhibit.ttl', format='turtle')
print('\nInstancias por clase:')
for clase, n in G.query('SELECT ?c (COUNT(?x) AS ?n) WHERE { ?x a ?c } GROUP BY ?c ORDER BY DESC(?n)'):
    print(f'  {str(clase).split("#")[-1].split("/")[-1]:30} {n}')

print('\nEjemplo: obras con título, técnica y alto (primeras 10)')
q = '''
PREFIX ox: <https://w3id.org/OntoExhibit#>
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?titulo ?tecnica ?alto WHERE {
  ?o a ox:Artwork ; rdfs:label ?titulo .
  OPTIONAL { ?o crm:P32_used_general_technique/rdfs:label ?tecnica }
  OPTIONAL { ?o crm:P43_has_dimension ?d . ?d crm:P2_has_type/rdfs:label "height" ; crm:P90_has_value ?alto }
} LIMIT 10'''
for fila in G.query(q):
    print('  ', [str(x) if x else '' for x in fila])

In [ ]:
# Cifras para el informe
import pandas as pd
n_cat = len(glob.glob(CARPETA_RDF + '/gold_por_catalogo/*.ttl'))
filas = [{'medida': 'Catálogos convertidos', 'n': n_cat}, {'medida': 'Tripletas', 'n': len(G)}]
for clase, n in G.query('SELECT ?c (COUNT(DISTINCT ?x) AS ?n) WHERE { ?x a ?c } GROUP BY ?c ORDER BY DESC(?n)'):
    filas.append({'medida': 'Instancias de ' + str(clase).split('#')[-1].split('/')[-1], 'n': int(n)})
resumen = pd.DataFrame(filas)
resumen.to_csv(CARPETA_RDF + '/resumen_rdf.csv', index=False)
with zipfile.ZipFile('/content/rdf_resultados.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for raiz, _, fs in os.walk(CARPETA_RDF):
        for f in fs:
            p = os.path.join(raiz, f); z.write(p, os.path.relpath(p, CARPETA_RDF))
files.download('/content/rdf_resultados.zip')
resumen